# MNIST Digit Classifier — Sketch-Ready

A CNN trained on MNIST that can **predict digits drawn in a Gradio sketchpad or uploaded as an image**, robust to:

- white or black backgrounds (auto color inversion)
- digits drawn small / off-center / in a corner (bbox crop + center-of-mass centering)
- thick, thin, or faint strokes
- stray dots / noise (connected-component cleanup)

**Key idea:** the exact same normalization used at training time (`/255.0 → [0, 1]`) is applied at inference, and every incoming sketch is reshaped into the MNIST format (28×28, black background, bright stroke, centered).

In [19]:
# !pip install tensorflow keras opencv-python matplotlib gradio

In [20]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.datasets import mnist

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load & prepare data

In [21]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training set:", x_train.shape, y_train.shape)
print("Testing set: ", x_test.shape, y_test.shape)

Training set: (60000, 28, 28) (60000,)
Testing set:  (10000, 28, 28) (10000,)


In [22]:
# Normalize to [0, 1] — inference must use EXACTLY the same scaling
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encode labels
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print("x_train range:", x_train.min(), "->", x_train.max())

x_train range: 0.0 -> 1.0


In [23]:
# Quick look at the data
fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(str(y_train[i]), fontsize=9)
    ax.axis("off")
plt.suptitle("Sample MNIST digits", y=1.05)
plt.tight_layout()
plt.show()

<Figure size 1200x150 with 10 Axes>

## 2. Model

A small CNN with **training-time augmentation** (random shift/zoom) so the model tolerates digits that are not perfectly centered — exactly what hand-drawn sketches look like. Augmentation layers are active only during `fit()`; predictions stay deterministic.

In [24]:
model = keras.Sequential(
    [
        layers.Input(shape=(28, 28, 1)),
        # Training-only augmentation (no effect at predict time)
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax"),
    ],
    name="mnist_cnn",
)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "mnist_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_translation              │ (None, 28, 28, 1)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Train

In [25]:
history = model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    epochs=5,
    batch_size=128,
    verbose=1,
)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.8686 - loss: 0.4136 - val_accuracy: 0.9789 - val_loss: 0.0641
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.9563 - loss: 0.1413 - val_accuracy: 0.9860 - val_loss: 0.0417
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.9685 - loss: 0.1025 - val_accuracy: 0.9884 - val_loss: 0.0331
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.9746 - loss: 0.0832 - val_accuracy: 0.9913 - val_loss: 0.0282
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.9768 - loss: 0.0746 - val_accuracy: 0.9886 - val_loss: 0.0318


In [26]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(history.history["accuracy"], label="train")
ax1.plot(history.history["val_accuracy"], label="val")
ax1.set_title("Accuracy")
ax1.set_xlabel("epoch")
ax1.legend()
ax2.plot(history.history["loss"], label="train")
ax2.plot(history.history["val_loss"], label="val")
ax2.set_title("Loss")
ax2.set_xlabel("epoch")
ax2.legend()
plt.tight_layout()
plt.show()

<Figure size 1100x350 with 2 Axes>

In [27]:
# Evaluate on the clean test set
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")

Test loss: 0.0318  |  Test accuracy: 0.9886 (98.86%)


In [28]:
# Save in native Keras format
model.save("mnist_model.keras")
print("Saved mnist_model.keras")

Saved mnist_model.keras


## 4. Sketch preprocessing

Every incoming drawing/photo goes through the **same pipeline** so it matches what the model saw during training:

1. Extract grayscale (handles RGBA sketchpad composites, RGB uploads, plain grayscale)
2. Auto-invert if the background is bright → MNIST convention (black background, bright stroke)
3. Otsu threshold + connected components → keep only the real stroke (drops noise/dots)
4. Bounding-box crop → resize to fit 20×20 (aspect preserved)
5. Place on a 28×28 canvas **centered by center of mass** (MNIST convention)
6. Scale to `[0, 1]` → shape `(1, 28, 28, 1)` — identical to training

In [29]:
def preprocess_digit_image(image_data):
    """Any sketch / photo / sketchpad dict -> (1, 28, 28, 1) float array in [0, 1]."""
    if image_data is None:
        return None

    # Gradio Sketchpad returns {"background", "layers", "composite"}
    if isinstance(image_data, dict):
        image_data = image_data.get("composite")
        if image_data is None:
            return None

    img = np.asarray(image_data)

    # --- 1. to grayscale ---
    if img.ndim == 3 and img.shape[-1] == 4:
        # composite RGBA over white, then grayscale
        rgb = img[:, :, :3].astype("float32")
        alpha = img[:, :, 3:4].astype("float32") / 255.0
        rgb = rgb * alpha + 255.0 * (1.0 - alpha)
        gray = cv2.cvtColor(np.clip(rgb, 0, 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
    elif img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.astype("uint8")

    if gray.size == 0:
        return None

    # --- 2. MNIST convention: black background, bright stroke ---
    if float(np.mean(gray)) > 127.0:
        gray = cv2.bitwise_not(gray)

    # --- 3. isolate the stroke (Otsu) and drop stray dots ---
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n_lbl > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        keep_area = 0.1 * areas.max()
        mask = np.where(
            (labels > 0) & (stats[labels, cv2.CC_STAT_AREA] >= keep_area), 255, 0
        ).astype(np.uint8)

    coords = cv2.findNonZero(mask)
    if coords is None:
        return None

    # --- 4. bounding-box crop with a small pad ---
    x, y, w, h = cv2.boundingRect(coords)
    pad = max(1, int(0.1 * max(w, h)))
    x0, y0 = max(0, x - pad), max(0, y - pad)
    x1, y1 = min(gray.shape[1], x + w + pad), min(gray.shape[0], y + h + pad)
    crop_gray = gray[y0:y1, x0:x1]
    crop_mask = mask[y0:y1, x0:x1]

    # fit inside 20x20 keeping aspect ratio (MNIST digits live in ~20x20)
    ch, cw = crop_gray.shape
    scale = 20.0 / max(ch, cw)
    nh, nw = max(1, int(round(ch * scale))), max(1, int(round(cw * scale)))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    digit = cv2.resize(crop_gray, (nw, nh), interpolation=interp)
    digit_mask = cv2.resize(crop_mask, (nw, nh), interpolation=interp)

    # --- 5. place on 28x28 canvas, centered by center of mass ---
    canvas = np.zeros((28, 28), dtype=np.float32)
    m = cv2.moments(digit_mask, binaryImage=True)
    if m["m00"] > 0:
        cx, cy = m["m10"] / m["m00"], m["m01"] / m["m00"]
    else:
        cx, cy = nw / 2.0, nh / 2.0
    tx = int(np.clip(round(14.0 - cx), 0, 28 - nw))
    ty = int(np.clip(round(14.0 - cy), 0, 28 - nh))
    canvas[ty:ty + nh, tx:tx + nw] = digit.astype("float32")

    # --- 6. same scaling as training ---
    canvas = np.clip(canvas, 0.0, 255.0) / 255.0
    return canvas.reshape(1, 28, 28, 1)


def predict_digit(image_data):
    """Return {digit_str: probability} for a sketchpad dict / ndarray."""
    processed = preprocess_digit_image(image_data)
    if processed is None:
        return {str(i): 0.0 for i in range(10)}
    probs = model.predict(processed, verbose=0)[0]
    return {str(i): float(p) for i, p in enumerate(probs)}

## 5. Sanity check — simulate different sketch styles

Take real test digits and re-draw them the way a sketchpad would: white canvas, corner/offset placement, thick/thin/faint strokes, noise, inverted colors — then run the full predict pipeline.

In [30]:
def draw_sketch(digit_img, style="center", canvas_size=280):
    """Simulate a sketchpad drawing of an MNIST digit."""
    canvas = np.full((canvas_size, canvas_size), 255, np.uint8)
    _, binary = cv2.threshold((digit_img * 255).astype(np.uint8), 100, 255, cv2.THRESH_BINARY)
    big = cv2.resize(binary, (180, 180), interpolation=cv2.INTER_NEAREST)
    mask = big > 0

    if style == "center":
        ox = oy = (canvas_size - 180) // 2
    elif style == "corner":
        ox, oy = 8, 8
    elif style == "offset":
        ox, oy = canvas_size - 190, canvas_size // 4
    else:
        ox = oy = (canvas_size - 180) // 2

    if style == "thick":
        mask = cv2.dilate(mask.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)

    region = canvas[oy:oy + 180, ox:ox + 180]
    region[mask] = 70 if style == "faint" else 0

    if style == "noisy":
        canvas = np.where(np.random.rand(*canvas.shape) < 0.01, 0, canvas).astype(np.uint8)
    if style == "inverted":
        canvas = cv2.bitwise_not(canvas)
    return canvas


styles = ["center", "corner", "offset", "thin", "thick", "faint", "noisy", "inverted"]
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(x_test), 60, replace=False)

n_test = 60
correct = {s: 0 for s in styles}
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

for si, style in enumerate(styles):
    for i in sample_idx:
        sketch = draw_sketch(x_test[i], style=style)
        # wrap as sketchpad-style RGBA dict to exercise the real code path
        rgba = np.dstack([sketch, sketch, sketch, np.full_like(sketch, 255)])
        probs = predict_digit({"composite": rgba, "layers": [], "background": None})
        pred = int(max(probs, key=probs.get))
        if pred == y_test[i]:
            correct[style] += 1

    # show one example per style
    example = draw_sketch(x_test[sample_idx[si]], style=style)
    rgba = np.dstack([example, example, example, np.full_like(example, 255)])
    processed = preprocess_digit_image({"composite": rgba})
    probs = predict_digit({"composite": rgba})
    pred = int(max(probs, key=probs.get))
    true = int(y_test[sample_idx[si]])

    axes[0, si].imshow(example, cmap="gray")
    axes[0, si].set_title(f"{style}\n{'OK' if pred == true else 'X'} pred={pred} true={true}", fontsize=8)
    axes[0, si].axis("off")
    axes[1, si].imshow(processed[0, :, :, 0], cmap="gray")
    axes[1, si].set_title("preprocessed", fontsize=7)
    axes[1, si].axis("off")
    axes[2, si].imshow(x_test[sample_idx[si]], cmap="gray")
    axes[2, si].set_title("true", fontsize=7)
    axes[2, si].axis("off")

plt.suptitle("Sketch styles -> preprocessing -> prediction", y=1.02)
plt.tight_layout()
plt.show()

print(f"Accuracy over {n_test} samples per style:")
for s in styles:
    print(f"  {s:12s} {correct[s]}/{n_test} = {correct[s] / n_test:.2%}")

<Figure size 1600x600 with 24 Axes>

Accuracy over 60 samples per style:
  center       58/60 = 96.67%
  corner       58/60 = 96.67%
  offset       58/60 = 96.67%
  thin         58/60 = 96.67%
  thick        58/60 = 96.67%
  faint        58/60 = 96.67%
  noisy        58/60 = 96.67%
  inverted     58/60 = 96.67%


## 6. Interactive Gradio app

Draw on the **Sketchpad** tab or **upload / photograph** a handwritten digit, then hit *Predict*.

Optional: if the model gets it wrong, select where you drew it, pick the true label, and press *Train on correction* — it fine-tunes gently (low learning rate, 3 steps) so the rest of the model is not wrecked, then re-saves `mnist_model.keras`.

In [31]:
import gradio as gr

# Gentle single-sample correction (small LR so we do not destroy trained weights)
feedback_optimizer = keras.optimizers.Adam(1e-3)
feedback_loss_fn = keras.losses.CategoricalCrossentropy()


def predict_interface(image_data):
    probs = predict_digit(image_data)
    if sum(probs.values()) == 0:
        return {"(draw something first)": 1.0}
    return probs


def predict_either(sketch, upload):
    filled_sketch = sketch is not None and (
        not isinstance(sketch, dict) or sketch.get("composite") is not None
    )
    data = sketch if filled_sketch else upload
    if data is None:
        return {"(draw or upload something first)": 1.0}
    return predict_interface(data)


def train_on_correction(image_data, correct_label):
    if image_data is None or correct_label in (None, ""):
        return "Draw a digit and select the true label first."
    processed = preprocess_digit_image(image_data)
    if processed is None:
        return "Could not read the drawing — try again."

    target = keras.utils.to_categorical(int(correct_label), 10)[None, :]
    first_loss = last_loss = None
    for step in range(3):
        with tf.GradientTape() as tape:
            preds = model(processed, training=True)
            loss = feedback_loss_fn(target, preds)
        grads = tape.gradient(loss, model.trainable_variables)
        feedback_optimizer.apply_gradients(zip(grads, model.trainable_variables))
        if step == 0:
            first_loss = float(loss)
        last_loss = float(loss)

    model.save("mnist_model.keras")
    probs = predict_digit(image_data)
    new_pred = max(probs, key=probs.get)
    return (
        f"Updated! loss {first_loss:.4f} -> {last_loss:.4f}. "
        f"Model now predicts: {new_pred} (confidence {probs[new_pred]:.0%}). Model re-saved."
    )


with gr.Blocks(title="MNIST Sketch Predictor") as demo:
    gr.Markdown("## MNIST Sketch Predictor\nDraw a digit (or upload a photo) and get a prediction.")
    with gr.Tabs():
        with gr.Tab("Sketchpad"):
            sketch_input = gr.Sketchpad(
                label="Draw here",
                type="numpy",
                brush=gr.Brush(colors=["#000000"], color_mode="fixed", default_size=18),
                canvas_size=(400, 400),
            )
        with gr.Tab("Upload / Webcam"):
            upload_input = gr.Image(
                label="Upload or capture a digit",
                type="numpy",
                sources=["upload", "webcam"],
                height=400,
            )

    with gr.Row():
        predict_btn = gr.Button("Predict", variant="primary")
        prediction_output = gr.Label(num_top_classes=3, label="Prediction")

    gr.Markdown("---")
    gr.Markdown("### Correct the model (optional)")
    with gr.Row():
        source_choice = gr.Radio(["Sketchpad", "Upload"], value="Sketchpad", label="Use drawing from")
        true_label = gr.Dropdown(choices=[str(i) for i in range(10)], label="True digit")
        feedback_status = gr.Textbox(label="Status", interactive=False)
    train_btn = gr.Button("Train on this correction", variant="stop")

    predict_btn.click(
        fn=predict_either,
        inputs=[sketch_input, upload_input],
        outputs=prediction_output,
    )
    train_btn.click(
        fn=lambda src, sk, up, lab: train_on_correction(
            sk if src == "Sketchpad" else up, lab
        ),
        inputs=[source_choice, sketch_input, upload_input, true_label],
        outputs=feedback_status,
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://2f48fdac6f0da9a585.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Re-evaluate after feedback

Run this after using the correction button to confirm the model is still healthy on the full test set.

In [32]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")

Test loss: 0.0318  |  Test accuracy: 0.9886 (98.86%)
